.# Feature Analysis
This notebook contains the exploratory data analysis (EDA) of the features created in `create_features.ipynb`. It includes feature coverage, value distributions, co-occurrence, descriptive statistics, unit consistency, feature importance, correlations, SBERT similarity analysis, embedding visualizations, and more.

**Prerequisite:** Run `create_features.ipynb` first to generate the saved data files.

## Load Data

In [1]:
import pandas as pd
import numpy as np
import re
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

# Set display options
pd.options.display.max_colwidth = 50
pd.options.display.max_columns = None
pd.options.display.float_format = '{:.2f}'.format

In [2]:
# Load saved data from create_features.ipynb
df_items_filtered = pd.read_pickle('../data/df_items_filtered.pkl')
df_items = pd.read_pickle('../data/df_items.pkl')
df_combined = pd.read_pickle('../data/df_combined.pkl')

# Load master_metadata
with open('../data/master_metadata.json') as f:
    master_metadata = json.load(f)

# Reconstruct all_keys from extracted_features
all_keys = set()
for feat_dict in df_items_filtered['extracted_features']:
    all_keys.update(feat_dict.keys())

print(f'df_items_filtered: {df_items_filtered.shape}')
print(f'df_items: {df_items.shape}')
print(f'df_combined: {df_combined.shape}')
print(f'Feature keys: {sorted(all_keys)}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/df_items_filtered.pkl'

## **EDA**

In [ ]:
# Explore extracted_features
from collections import Counter

key_counts = Counter()

for feat_dict in df_items_filtered['extracted_features']:
  for key in feat_dict:
      key_counts[key] += 1

for key, count in key_counts.most_common():
    print(f"  {key}: {count:,}")

In [ ]:
# Features available across categories (in how many category a feature appears)
from collections import Counter

field_counts = Counter()
field_cats = {}

for cat, fields in master_metadata.items():
    for field in fields.keys():
        field_counts[field] += 1
        field_cats.setdefault(field, []).append(cat)

print(f"{'Field':<28} {'# Categories':>14}")
print('=' * 44)
for field, count in field_counts.most_common():
    print(f"  {field:<26} {count:>3} / {len(master_metadata)}")

In [ ]:
# Unique extracted feature values per cat_3 from the data
feature_cols = sorted([col for col in df_items_filtered.columns if col in all_keys])

for cat3, group in df_items_filtered.groupby('cat_3'):
    print(f"\n{'='*70}")
    print(f"{cat3} ({len(group):,} items)")
    print(f"{'='*70}")
    for col in feature_cols:
        unique_vals = group[col].dropna().unique()
        if len(unique_vals) == 0:
            continue
        print(f"\n  {col} ({len(unique_vals)} unique values):")
        print(f"    {sorted(unique_vals)}")

 1. Feature Coverage Analysis. <br> 2. Value Distribution Analysis. <br> 3. Feature co-occurrence <br> 4. Description of numeric features >br? 5, Unit consistency (e.g., % inch vs cm) <br> 6. Feature importance (vary across items, not too sparse) <br> 7. Feature correlation

In [ ]:
# 1. Feature Coverage Analysis
# How many items have each feature? Overall and by category.

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

all_feature_cols = sorted([col for col in df_items_filtered.columns if col in all_keys])

# Overall coverage
coverage = df_items_filtered[all_feature_cols].notna().mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 8))
coverage.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Coverage (%)')
ax.set_title('Feature Coverage: % of Items with Non-Null Values')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(coverage):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()

# Coverage by cat_3 (top 10 categories by item count)
top_cats = df_items_filtered['cat_3'].value_counts().head(10).index
coverage_by_cat = df_items_filtered[df_items_filtered['cat_3'].isin(top_cats)].groupby('cat_3')[all_feature_cols].apply(
    lambda x: x.notna().mean() * 100
)

fig, ax = plt.subplots(figsize=(14, 8))
coverage_by_cat[coverage.index].plot(kind='barh', ax=ax, legend=True)
ax.set_xlabel('Coverage (%)')
ax.set_title('Feature Coverage by Top 10 Categories')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# 2a. Value Distribution: Categorical Features

categorical_features = ['Brand', 'Color', 'Features', 'Material', 'Product_Type',
                        'Scent', 'Shape', 'Shape_Style', 'Size', 'Sub_Type', 'Theme']

n_cols = 3
n_rows = (len(categorical_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    top_vals = df_items_filtered[col].value_counts().head(15)
    if len(top_vals) > 0:
        top_vals.plot(kind='barh', ax=axes[i], color='steelblue')
        axes[i].set_title(f'{col} (top 15)', fontsize=10)
        axes[i].tick_params(axis='y', labelsize=8)
    else:
        axes[i].set_title(f'{col} (no data)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Value Distribution: Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 2b. Value Distribution: Numeric Features

numeric_features = [
    'bar_pressure_numeric', 'capacity_numeric', 'capacity_cups_numeric',
    'capacity_volume_numeric', 'density_weight_lb', 'piece_count_numeric',
    'pocket_depth_in', 'power_rating_w', 'stage_count_numeric',
    'thread_count_numeric', 'voltage_numeric', 'weight_numeric',
    'dimension_1', 'dimension_2', 'dimension_3'
]
numeric_features = [c for c in numeric_features if c in df_items_filtered.columns and df_items_filtered[c].notna().sum() > 0]

n_cols = 3
n_rows = (len(numeric_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    data = df_items_filtered[col].dropna()
    # Clip outliers for better visualization (1st-99th percentile)
    q01, q99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data[(data >= q01) & (data <= q99)]
    axes[i].hist(data_clipped, bins=50, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col} (n={len(data):,})', fontsize=10)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Value Distribution: Numeric Features (1st-99th percentile)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Feature Co-occurrence Analysis
import seaborn as sns

# Binary presence matrix for all extracted features
presence = df_items_filtered[all_feature_cols].notna().astype(int)

# Co-occurrence matrix
cooccurrence = presence.T.dot(presence)

# Normalize by the minimum count of each pair
diag = np.diag(cooccurrence).astype(float)
min_matrix = np.minimum(diag[:, None], diag[None, :])
min_matrix[min_matrix == 0] = 1
cooccurrence_pct = (cooccurrence.values / min_matrix) * 100
cooccurrence_pct_df = pd.DataFrame(cooccurrence_pct, index=cooccurrence.index, columns=cooccurrence.columns)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cooccurrence_pct_df, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, square=True, cbar_kws={'label': '% co-occurrence (of smaller feature)'})
ax.set_title('Feature Co-occurrence (% of the less common feature)')
plt.tight_layout()
plt.show()

# Top co-occurring pairs
pairs = []
for i, f1 in enumerate(all_feature_cols):
    for j, f2 in enumerate(all_feature_cols):
        if i < j:
            both = presence[f1].mul(presence[f2]).sum()
            if both > 0:
                pairs.append((f1, f2, both, cooccurrence_pct_df.loc[f1, f2]))

pairs_df = pd.DataFrame(pairs, columns=['Feature_1', 'Feature_2', 'Co-occur_Count', 'Co-occur_%'])
print("Top 15 co-occurring feature pairs:")
print(pairs_df.sort_values('Co-occur_Count', ascending=False).head(15).to_string(index=False))

In [ ]:
# 4. Descriptive Statistics of Numeric Features by cat_2

numeric_cols = [
    'bar_pressure_numeric', 'capacity_numeric', 'capacity_cups_numeric',
    'capacity_volume_numeric', 'density_weight_lb', 'piece_count_numeric',
    'pocket_depth_in', 'power_rating_w', 'stage_count_numeric',
    'thread_count_numeric', 'voltage_numeric', 'weight_numeric',
    'dimension_1', 'dimension_2', 'dimension_3'
]
numeric_cols = [c for c in numeric_cols if c in df_items_filtered.columns]

for cat2 in sorted(df_items_filtered['cat_2'].unique()):
    subset = df_items_filtered[df_items_filtered['cat_2'] == cat2]
    desc = subset[numeric_cols].describe().T
    desc['non_null'] = subset[numeric_cols].notna().sum()
    desc['missing_%'] = (subset[numeric_cols].isna().sum() / len(subset) * 100).round(1)
    desc['skewness'] = subset[numeric_cols].skew().round(2)
    desc['kurtosis'] = subset[numeric_cols].kurtosis().round(2)
    print(f"\n{'='*80}")
    print(f"{cat2} ({len(subset):,} items)")
    print(f"{'='*80}")
    print(desc[['non_null', 'missing_%', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']].to_string())

In [ ]:
# 4b. Feature Availability (%) by cat_2 — Categorical and Numeric

all_feature_cols_analysis = [
    'Bar_Pressure', 'Brand', 'Capacity', 'Capacity_Cups', 'Capacity_Volume',
    'Color', 'Density_Weight', 'Dimensions', 'Features', 'Filter_Rating',
    'Material', 'Part_Number', 'Piece_Count', 'Pocket_Depth', 'Power_Rating',
    'Product_Type', 'Scent', 'Shape', 'Shape_Style', 'Size', 'Stage_Count',
    'Sub_Type', 'Theme', 'Thread_Count', 'Voltage', 'Weight'
]
all_feature_cols_analysis = [c for c in all_feature_cols_analysis if c in df_items_filtered.columns]

availability = df_items_filtered.groupby('cat_2')[all_feature_cols_analysis].apply(
    lambda x: (x.notna().sum() / len(x) * 100).round(1)
)

print("Feature Availability (%) by cat_2:\n")
print(availability.to_string())

In [ ]:
# 4c. Numeric Features by cat_2

numeric_cols = [
    'bar_pressure_numeric', 'capacity_numeric', 'capacity_cups_numeric',
    'capacity_volume_numeric', 'density_weight_lb', 'piece_count_numeric',
    'pocket_depth_in', 'power_rating_w', 'stage_count_numeric',
    'thread_count_numeric', 'voltage_numeric', 'weight_numeric',
    'dimension_1', 'dimension_2', 'dimension_3'
]
numeric_cols = [c for c in numeric_cols if c in df_items_filtered.columns]

for cat2 in sorted(df_items_filtered['cat_2'].unique()):
    subset = df_items_filtered[df_items_filtered['cat_2'] == cat2]
    print(f"\n{'='*80}")
    print(f"{cat2} ({len(subset):,} items)")
    print(f"{'='*80}")
    rows = []
    for col in numeric_cols:
        non_null = subset[col].notna().sum()
        available_pct = round((non_null / len(subset)) * 100, 1)
        n_unique = subset[col].nunique(dropna=True)
        top_vals = subset[col].value_counts().head(5)
        top_str = ', '.join([f'{v} ({c:,})' for v, c in top_vals.items()]) if len(top_vals) > 0 else '-'
        rows.append({
            'Feature': col,
            'Non_Null': non_null,
            'Available_%': available_pct,
            'Unique': n_unique,
            'Top_5_Values': top_str
        })
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 4d. Categorical Features by cat_2

categorical_cols = ['Brand', 'Color', 'Features', 'Filter_Rating', 'Material',
                    'Part_Number', 'Product_Type', 'Scent', 'Shape', 'Shape_Style',
                    'Size', 'Sub_Type', 'Theme']
categorical_cols = [c for c in categorical_cols if c in df_items_filtered.columns]

for cat2 in sorted(df_items_filtered['cat_2'].unique()):
    subset = df_items_filtered[df_items_filtered['cat_2'] == cat2]
    print(f"\n{'='*80}")
    print(f"{cat2} ({len(subset):,} items)")
    print(f"{'='*80}")
    rows = []
    for col in categorical_cols:
        non_null = subset[col].notna().sum()
        available_pct = round((non_null / len(subset)) * 100, 1)
        n_unique = subset[col].nunique(dropna=True)
        top_vals = subset[col].value_counts().head(5)
        top_str = ', '.join([f'{v} ({c:,})' for v, c in top_vals.items()]) if len(top_vals) > 0 else '-'
        rows.append({
            'Feature': col,
            'Non_Null': non_null,
            'Available_%': available_pct,
            'Unique': n_unique,
            'Top_5_Values': top_str
        })
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 5. Unit Consistency Analysis

unit_cols_to_check = {
    'capacity_unit': 'Capacity',
    'capacity_volume_unit': 'Capacity_Volume',
    'piece_count_unit': 'Piece_Count',
    'thread_count_unit': 'Thread_Count',
    'weight_unit': 'Weight',
    'dimension_unit': 'Dimensions',
}
unit_cols_to_check = {k: v for k, v in unit_cols_to_check.items() if k in df_items_filtered.columns}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, (col, label) in enumerate(unit_cols_to_check.items()):
    counts = df_items_filtered[col].value_counts()
    if len(counts) > 0:
        counts.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='white')
        axes[i].set_title(f'{label} units', fontsize=10)
        axes[i].tick_params(axis='x', rotation=45, labelsize=8)
        for j, v in enumerate(counts):
            axes[i].text(j, v, f'{v:,}', ha='center', va='bottom', fontsize=7)
    else:
        axes[i].set_title(f'{label} (no data)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Unit Consistency: Distribution of Units per Feature', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Summary table
print("\nUnit distribution summary:")
for col, label in unit_cols_to_check.items():
    counts = df_items_filtered[col].value_counts()
    total = counts.sum()
    print(f"\n{label} ({total:,} items with units):")
    for unit, count in counts.items():
        print(f"  {unit}: {count:,} ({count/total*100:.1f}%)")

In [ ]:
# 6. Feature Importance: Variance and Sparsity
from scipy.stats import entropy as scipy_entropy

importance_rows = []
for col in all_feature_cols:
    non_null = df_items_filtered[col].notna().sum()
    coverage_pct = non_null / len(df_items_filtered) * 100
    n_unique = df_items_filtered[col].nunique(dropna=True)
    
    if n_unique > 1:
        val_counts = df_items_filtered[col].value_counts(normalize=True)
        ent = scipy_entropy(val_counts, base=2)
        max_ent = np.log2(n_unique)
        norm_entropy = ent / max_ent if max_ent > 0 else 0
    else:
        ent = 0
        norm_entropy = 0
    
    importance_rows.append({
        'Feature': col,
        'Coverage_%': round(coverage_pct, 1),
        'Unique_Values': n_unique,
        'Entropy': round(ent, 2),
        'Normalized_Entropy': round(norm_entropy, 3),
    })

importance_df = pd.DataFrame(importance_rows).sort_values('Coverage_%', ascending=False)

print("Feature Importance: Variance and Sparsity")
print("(Higher entropy = more variation, higher coverage = less sparse)\n")
print(importance_df.to_string(index=False))

# Scatter plot: coverage vs entropy
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(importance_df['Coverage_%'], importance_df['Normalized_Entropy'],
           s=importance_df['Unique_Values'].clip(upper=500) * 0.5, alpha=0.6, color='steelblue')
for _, row in importance_df.iterrows():
    ax.annotate(row['Feature'], (row['Coverage_%'], row['Normalized_Entropy']),
                fontsize=7, ha='center', va='bottom')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Normalized Entropy')
ax.set_title('Feature Usefulness: Coverage vs Variation\n(bubble size = unique values)')
ax.axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% coverage threshold')
ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='0.5 entropy threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7. Feature Correlation (Numeric Features)

numeric_cols_corr = [
    'capacity_volume_numeric', 'piece_count_numeric', 'thread_count_numeric',
    'weight_numeric', 'density_weight_lb', 'pocket_depth_in', 'power_rating_w',
    'voltage_numeric', 'bar_pressure_numeric', 'capacity_numeric',
    'capacity_cups_numeric', 'stage_count_numeric',
    'dimension_1', 'dimension_2', 'dimension_3'
]
numeric_cols_corr = [c for c in numeric_cols_corr if c in df_items_filtered.columns and df_items_filtered[c].notna().sum() > 100]

corr_matrix = df_items_filtered[numeric_cols_corr].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Feature Correlation Matrix (Numeric Features)')
plt.tight_layout()
plt.show()

# Top correlated pairs
corr_pairs = []
for i, c1 in enumerate(numeric_cols_corr):
    for j, c2 in enumerate(numeric_cols_corr):
        if i < j:
            r = corr_matrix.loc[c1, c2]
            n_both = df_items_filtered[[c1, c2]].dropna().shape[0]
            if n_both > 100:
                corr_pairs.append((c1, c2, round(r, 3), n_both))

corr_pairs_df = pd.DataFrame(corr_pairs, columns=['Feature_1', 'Feature_2', 'Correlation', 'N_Both'])
print("Top correlated feature pairs (|r| > 0.3, n > 100):")
top_corr = corr_pairs_df[corr_pairs_df['Correlation'].abs() > 0.3].sort_values('Correlation', key=abs, ascending=False)
if len(top_corr) > 0:
    print(top_corr.to_string(index=False))
else:
    print("No pairs with |r| > 0.3 and sufficient overlap.")

In [ ]:
# Unique values for each feature variable
feature_cols = sorted([col for col in df_items_filtered.columns if col in all_keys])

for col in feature_cols:
    unique_vals = df_items_filtered[col].dropna().unique()
    if len(unique_vals) == 0:
        continue
    print(f"{col} ({len(unique_vals)} unique values):")
    print(f"  {sorted(unique_vals)[:30]}")
    print()

In [ ]:
# Unique ASINs by cat_4 for selected cat_3 categories
selected_cat3 = [
    # Kitchen & Dining
    'Dining & Entertaining', 'Kitchen Utensils & Gadgets', 'Storage & Organization',
    'Kitchen & Table Linens', 'Bakeware', 'Cookware', 'Cutlery & Knife Accessories',
    'Small Appliances', 'Travel & To-Go Drinkware', 'Coffee, Tea & Espresso',
    'Small Appliance Parts & Accessories', 'Water Coolers & Filters',
    'Home Brewing & Wine Making', 'Wine Accessories',
    # Home Decor
    'Home Dcor Accents', 'Candles & Holders', 'Area Rugs, Runners & Pads',
    'Window Treatments', 'Clocks', 'Picture Frames', 'Home Fragrance',
    'Artificial Plants & Flowers', 'Vases', 'Kids\' Room Dcor',
    'Indoor Fountains & Accessories', 'Mirrors', 'Tapestries', 'Slipcovers',
    'Window Treatment Hardware', 'Photo Albums & Accessories',
    'Window Stickers & Films', 'Gift Baskets', 'Oil Lamps & Accessories',
    'Vase Fillers', 'Draft Stoppers',
    # Bedding
    'Decorative Pillows, Inserts & Covers', 'Sheets & Pillowcases',
    'Blankets & Throws', 'Comforters & Sets', 'Duvets, Covers & Sets',
    'Kids\' Bedding', 'Bed Pillows & Positioners', 'Quilts & Sets',
    'Mattress Pads & Protectors', 'Bed Skirts', 'Bedspreads, Coverlets & Sets',
    'Bedding Sets & Collections', 'Mattress Toppers', 'Air Mattresses & Accessories',
    'Bedding Accessories', 'Bed Canopies & Drapes', 'Bed Runners & Scarves',
    # Wall Art
    'Posters & Prints', 'Paintings', 'Photographs',
    # Furniture
    'Living Room Furniture', 'Bedroom Furniture', 'Game & Recreation Room Furniture',
    'Kitchen & Dining Room Furniture', 'Home Office Furniture', 'Kids\' Furniture',
    'Accent Furniture', 'Entryway Furniture', 'Replacement Parts', 'Bathroom Furniture',
    # Bath
    'Bathroom Accessories', 'Towels', 'Bath Rugs', 'Kids\' Bath',
]

rows = []
for cat3 in selected_cat3:
    subset = df_items_filtered[df_items_filtered['cat_3'] == cat3]
    cat4_counts = subset.groupby('cat_4', dropna=False)['asin'].nunique().sort_values(ascending=False)
    print(f"\n{'='*60}")
    print(f"{cat3} ({subset['asin'].nunique():,} unique ASINs, {len(cat4_counts)} cat_4 values)")
    print(f"{'='*60}")
    print(cat4_counts.to_string())
    for cat4, count in cat4_counts.items():
        rows.append({'cat_3': cat3, 'cat_4': cat4, 'unique_asins': count})

cat3_cat4_df = pd.DataFrame(rows)

In [ ]:
# Distribution of single-variable numeric features
single_var_cols = {
    'Bar_Pressure': 'bar_pressure_numeric',
    'Capacity_Cups': 'capacity_cups_numeric',
    'Density_Weight': 'density_weight_lb',
    'Pocket_Depth': 'pocket_depth_in',
    'Power_Rating': 'power_rating_w',
    'Stage_Count': 'stage_count_numeric',
    'Voltage': 'voltage_numeric',
}

cols_with_data = {k: v for k, v in single_var_cols.items() if df_items_filtered[v].notna().sum() > 0}

fig, axes = plt.subplots(len(cols_with_data), 1, figsize=(12, 4 * len(cols_with_data)))
if len(cols_with_data) == 1:
    axes = [axes]

for i, (label, col) in enumerate(cols_with_data.items()):
    data = df_items_filtered[col].dropna()
    q01, q99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data[(data >= q01) & (data <= q99)]
    axes[i].hist(data_clipped, bins=50, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{label} → {col} (n={len(data):,})', fontsize=11)
    axes[i].set_xlabel(col)

plt.suptitle('Distribution of Single-Variable Numeric Features (1st-99th percentile)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Summary statistics
for label, col in single_var_cols.items():
    data = df_items_filtered[col].dropna()
    if len(data) == 0:
        print(f"\n{label} → {col}: no data")
    else:
        print(f"\n{label} → {col} (n={len(data):,}):")
        print(f"  {data.describe().to_string()}")
        print(f"  Top 10 values: {data.value_counts().head(10).to_dict()}")

In [ ]:
# Distribution of dual-variable numeric features (numeric + unit)
dual_var_cols = {
    'Capacity': 'capacity_numeric',
    'Capacity_Volume': 'capacity_volume_numeric',
    'Piece_Count': 'piece_count_numeric',
    'Thread_Count': 'thread_count_numeric',
    'Weight': 'weight_numeric',
}

cols_with_data = {k: v for k, v in dual_var_cols.items() if df_items_filtered[v].notna().sum() > 0}

fig, axes = plt.subplots(len(cols_with_data), 1, figsize=(12, 4 * len(cols_with_data)))
if len(cols_with_data) == 1:
    axes = [axes]

for i, (label, col) in enumerate(cols_with_data.items()):
    data = df_items_filtered[col].dropna()
    q01, q99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data[(data >= q01) & (data <= q99)]
    axes[i].hist(data_clipped, bins=50, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{label} → {col} (n={len(data):,})', fontsize=11)
    axes[i].set_xlabel(col)

plt.suptitle('Distribution of Dual-Variable Numeric Features (1st-99th percentile)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Summary statistics
for label, col in dual_var_cols.items():
    data = df_items_filtered[col].dropna()
    if len(data) == 0:
        print(f"\n{label} → {col}: no data")
    else:
        print(f"\n{label} → {col} (n={len(data):,}):")
        print(f"  {data.describe().to_string()}")
        print(f"  Top 10 values: {data.value_counts().head(10).to_dict()}")

In [ ]:
# Unique values for the unit (categorical) columns of dual-variable features
unit_cols = {
    'Capacity': 'capacity_unit',
    'Capacity_Volume': 'capacity_volume_unit',
    'Piece_Count': 'piece_count_unit',
    'Thread_Count': 'thread_count_unit',
    'Weight': 'weight_unit',
}

for label, col in unit_cols.items():
    unique_vals = sorted(df_items_filtered[col].dropna().unique())
    count_by_unit = df_items_filtered[col].value_counts(dropna=False).sort_values(ascending=False)
    print(f"\n{label} → {col} ({len(unique_vals)} unique values):")
    print(f"  Values: {unique_vals}")
    print(f"  Distribution:")
    for unit, count in count_by_unit.items():
        pct = count / len(df_items_filtered) * 100
        print(f"    {unit}: {count:,} ({pct:.1f}%)")

In [ ]:
# Distribution of Dimension features
dimension_cols = {
    'Dimensions (dim1)': 'dimension_1',
    'Dimensions (dim2)': 'dimension_2',
    'Dimensions (dim3)': 'dimension_3',
}

cols_with_data = {k: v for k, v in dimension_cols.items() if df_items_filtered[v].notna().sum() > 0}

fig, axes = plt.subplots(len(cols_with_data), 1, figsize=(12, 4 * len(cols_with_data)))
if len(cols_with_data) == 1:
    axes = [axes]

for i, (label, col) in enumerate(cols_with_data.items()):
    data = df_items_filtered[col].dropna()
    q01, q99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data[(data >= q01) & (data <= q99)]
    axes[i].hist(data_clipped, bins=50, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{label} → {col} (n={len(data):,})', fontsize=11)
    axes[i].set_xlabel(col)

plt.suptitle('Distribution of Dimension Features (1st-99th percentile)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Summary statistics
for label, col in dimension_cols.items():
    data = df_items_filtered[col].dropna()
    if len(data) == 0:
        print(f"\n{label} → {col}: no data")
    else:
        print(f"\n{label} → {col} (n={len(data):,}):")
        print(f"  {data.describe().to_string()}")
        print(f"  Top 10 values: {data.value_counts().head(10).to_dict()}")

In [ ]:
# Unique values for dimension_unit
unique_vals = sorted(df_items_filtered['dimension_unit'].dropna().unique())
count_by_unit = df_items_filtered['dimension_unit'].value_counts(dropna=False).sort_values(ascending=False)

print(f"Dimensions → dimension_unit ({len(unique_vals)} unique values):")
print(f"  Values: {unique_vals}")
print(f"  Distribution:")
for unit, count in count_by_unit.items():
    pct = count / len(df_items_filtered) * 100
    print(f"    {unit}: {count:,} ({pct:.1f}%)")

In [ ]:
 df_items_filtered[df_items_filtered['cat_2'] == 'Bedding']['Size'].unique()

In [ ]:
# Boxplots: Dimensions by Size for Bedding
import seaborn as sns

bedding = df_items_filtered[
    (df_items_filtered['cat_2'] == 'Bedding') &
    (df_items_filtered['dimension_unit'] == 'in') &
    (df_items_filtered['Size'].notna())
].copy()

dim_cols = ['dimension_1', 'dimension_2', 'dimension_3']

# Compute shared y-axis limits across all three dimensions
all_vals = []
for col in dim_cols:
    vals = bedding[col].dropna()
    q01, q99 = vals.quantile(0.01), vals.quantile(0.99)
    all_vals.append(vals[(vals >= q01) & (vals <= q99)])
y_min = min(v.min() for v in all_vals if len(v) > 0)
y_max = max(v.max() for v in all_vals if len(v) > 0)
y_pad = (y_max - y_min) * 0.05

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for i, col in enumerate(dim_cols):
    data = bedding[bedding[col].notna()]
    # Clip outliers for better visualization
    q01, q99 = data[col].quantile(0.01), data[col].quantile(0.99)
    data = data[(data[col] >= q01) & (data[col] <= q99)]
    
    sns.boxplot(data=data, x='Size', y=col, ax=axes[i], showfliers=False,
                order=data['Size'].value_counts().index)
    axes[i].set_title(f'{col} by Size', fontsize=12)
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_ylim(y_min - y_pad, y_max + y_pad)

plt.suptitle('Bedding: Dimensions by Size', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_items_filtered['dimension_unit'].unique()

In [ ]:
df_items_filtered[
    (df_items_filtered['dimension_1'].isna() == False) &
    (df_items_filtered['dimension_unit'].isna() == True)
]

In [ ]:
pd.options.display.max_colwidth = 50
df_items_filtered[
    (df_items_filtered['cat_2'] == 'Bedding') &
    (df_items_filtered['Size'] == 'single') &
    (df_items_filtered['dimension_2'] > 175)
]['description_cleaned']

In [ ]:
# Check number of unique values in unit columns
unit_cols = [
    'bar_pressure_unit', 'capacity_unit', 'capacity_cups_unit',
    'capacity_volume_unit', 'density_weight_unit', 'piece_count_unit',
    'pocket_depth_unit', 'power_rating_unit', 'stage_count_unit',
    'thread_count_unit', 'voltage_unit', 'weight_unit'
]

for col in unit_cols:
    unique_vals = df_items_filtered[col].dropna().unique()
    print(f"{col}: {len(unique_vals)} unique values → {sorted(unique_vals)}")

**Analyze the "General" Features.**

The general features are the following:dimension_1, dimension_2, dimension_3, weight_numeric, material, color, product_type, features, piece_count_numeric

In [ ]:
dimension_cols = ['dimension_1', 'dimension_2', 'dimension_3']

for cat2 in sorted(df_items_filtered['cat_2'].unique()):
    subset = df_items_filtered[df_items_filtered['cat_2'] == cat2]
    desc = subset[dimension_cols].describe().T
    desc['non_null'] = subset[dimension_cols].notna().sum()
    desc['missing_%'] = (subset[dimension_cols].isna().sum() / len(subset) * 100).round(1)
    desc['skewness'] = subset[dimension_cols].skew().round(2)
    desc['kurtosis'] = subset[dimension_cols].kurtosis().round(2)
    print(f"\n{'='*80}")
    print(f"{cat2} ({len(subset):,} items)")
    print(f"{'='*80}")
    print(desc[['non_null', 'missing_%', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']].to_string())

In [ ]:
# Distribution of dimension_1, dimension_2, dimension_3 per cat_2
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

dimension_cols = ['dimension_1', 'dimension_2', 'dimension_3']
cat2_values = sorted(df_items_filtered['cat_2'].unique())

for dim_col in dimension_cols:
    fig, axes = plt.subplots(len(cat2_values), 1, figsize=(12, 4 * len(cat2_values)))
    for i, cat2 in enumerate(cat2_values):
        subset = df_items_filtered[df_items_filtered['cat_2'] == cat2][dim_col].dropna()
        if len(subset) == 0:
            axes[i].set_title(f'{cat2} — {dim_col} (no data)')
            continue
        axes[i].hist(subset, bins=50, color='steelblue', edgecolor='white')
        axes[i].set_title(f'{cat2} — {dim_col} (n={len(subset):,})', fontsize=11)
        axes[i].set_xlabel(dim_col)
        axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        axes[i].xaxis.set_major_locator(mticker.MaxNLocator(nbins=20))
        axes[i].yaxis.set_major_locator(mticker.MaxNLocator(nbins=10))
        axes[i].tick_params(axis='x', rotation=45)
    plt.suptitle(f'Distribution of {dim_col} by cat_2', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Unique Material values by cat_2
for cat2 in sorted(df_items_filtered['cat_2'].unique()):
    subset = df_items_filtered[df_items_filtered['cat_2'] == cat2]['Material'].dropna().unique()
    if len(subset) == 0:
        continue
    print(f"{cat2} ({len(subset)} unique values):")
    for v in sorted(subset):
        print(f"  {v}")
    print()

In [ ]:
pd.options.display.max_colwidth = None

In [ ]:
df_items_filtered[(df_items_filtered['cat_2'] == 'Bath') &
                  (df_items_filtered['dimension_1'] > 1000)]['title_cleaned']
# description_cleaned
# ['all_text_cleaned']

## SBERT Embedding Analysis

In [ ]:
# Prepare embedding matrix for similarity analysis
embedding_matrix = np.stack(df_items_filtered['title_embedding'].values)
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
embedding_matrix_norm = embedding_matrix / norms
asins = df_items_filtered['asin'].values

print(f'Embedding matrix: {embedding_matrix.shape}')

In [ ]:
# Return the top10 similar items for a given asin
embedding_matrix = np.stack(df_items_filtered['title_embedding'].values)
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
embedding_matrix_norm = embedding_matrix / norms
asins = df_items_filtered['asin'].values

def get_similar_items(asin, n=10):
    """Return top N most similar items for a given ASIN using dot product (cosine similarity)."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]
    
    # Dot product against all items
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm.T
    scores[idx] = -1  # exclude self
    
    # Top N
    top_indices = np.argsort(scores)[-n:][::-1]
    
    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_indices].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'cat_5': df_items_filtered.iloc[top_indices]['cat_5'].values,
        'cat_6': df_items_filtered.iloc[top_indices]['cat_6'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })
    
    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin}\n")
    
    return results

# Set display option to show full content of columns
pd.options.display.max_colwidth = None
get_similar_items('B00029TCRG', n=10)

In [ ]:
# Return the top N similar items, prioritizing same cat_3 (with cat_4 boost)
def get_similar_items_by_category(asin, n=10, cat3_boost=0.1, cat4_boost=0.05):
    """Return top N most similar items, boosting same cat_3 and cat_4 matches."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]

    query_cat3 = df_items_filtered.iloc[idx]['cat_3']
    query_cat4 = df_items_filtered.iloc[idx]['cat_4']

    # Cosine similarity against all items
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm.T
    scores[idx] = -1  # exclude self

    # Boost same cat_3, and additionally boost same cat_4
    boosted_scores = scores.copy()
    cat3_vals = df_items_filtered['cat_3'].values
    cat4_vals = df_items_filtered['cat_4'].values

    same_cat3 = cat3_vals == query_cat3
    same_cat4 = same_cat3 & (cat4_vals == query_cat4)

    boosted_scores[same_cat3] += cat3_boost
    boosted_scores[same_cat4] += cat4_boost

    # Top N by boosted scores
    top_indices = np.argsort(boosted_scores)[-n:][::-1]

    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_indices].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'cat_5': df_items_filtered.iloc[top_indices]['cat_5'].values,
        'cat_6': df_items_filtered.iloc[top_indices]['cat_6'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })

    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin}")
    print(f"cat_3: {query_cat3} | cat_4: {query_cat4}\n")

    return results

pd.options.display.max_colwidth = None
get_similar_items_by_category('B00029TCRG', n=10)

In [ ]:
df_items_filtered[df_items_filtered['asin'] == 'B00029TCRG'][['cat_4', 'title']]

In [ ]:
pd.options.display.max_colwidth = 50

In [ ]:
# Return the top N most similar items within the same cat_3 category
def get_similar_items_same_cat3(asin, n=10):
    """Return top N most similar items for a given ASIN, restricted to the same cat_3."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]

    query_cat3 = df_items_filtered.iloc[idx]['cat_3']

    # Get indices of items in the same cat_3
    same_cat3_mask = df_items_filtered['cat_3'].values == query_cat3
    same_cat3_indices = np.where(same_cat3_mask)[0]

    # Cosine similarity against same-category items only
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm[same_cat3_indices].T
    
    # Exclude self
    self_pos = np.where(same_cat3_indices == idx)[0]
    if len(self_pos) > 0:
        scores[self_pos[0]] = -1

    # Top N
    top_n = min(n, len(scores))
    top_local = np.argsort(scores)[-top_n:][::-1]
    top_indices = same_cat3_indices[top_local]

    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_local].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })

    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin} | cat_3: {query_cat3}")
    print(f"Same-category items: {same_cat3_mask.sum():,}\n")

    return results

pd.options.display.max_colwidth = None
get_similar_items_same_cat3('B00029TCRG', n=10)

In [ ]:
# Return the top N most similar items within the same cat_3 but different cat_4
def get_similar_items_same_cat3_diff_cat4(asin, n=10):
    """Return top N most similar items for a given ASIN, same cat_3 but different cat_4."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]

    query_cat3 = df_items_filtered.iloc[idx]['cat_3']
    query_cat4 = df_items_filtered.iloc[idx]['cat_4']

    # Get indices of items in the same cat_3 but different cat_4
    mask = (df_items_filtered['cat_3'].values == query_cat3) & (df_items_filtered['cat_4'].values != query_cat4)
    filtered_indices = np.where(mask)[0]

    if len(filtered_indices) == 0:
        print(f"No items found in cat_3='{query_cat3}' with a different cat_4 than '{query_cat4}'.")
        return None

    # Cosine similarity against filtered items only
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm[filtered_indices].T

    # Top N
    top_n = min(n, len(scores))
    top_local = np.argsort(scores)[-top_n:][::-1]
    top_indices = filtered_indices[top_local]

    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_local].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })

    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin} | cat_3: {query_cat3} | cat_4: {query_cat4}")
    print(f"Same cat_3, different cat_4 items: {mask.sum():,}\n")

    return results

get_similar_items_same_cat3_diff_cat4('B00029TCRG', n=10)

In [ ]:
# Return the top N most similar items within the same cat_3 but different cat_4 and product_type
def get_similar_items_same_cat3_diff_cat4_product_type(asin, n=10):
    """Return top N most similar items for a given ASIN, same cat_3 but different cat_4 and Product_Type."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]

    query_cat3 = df_items_filtered.iloc[idx]['cat_3']
    query_cat4 = df_items_filtered.iloc[idx]['cat_4']
    query_product_type = df_items_filtered.iloc[idx]['Product_Type']

    # Get indices of items in the same cat_3 but different cat_4 and Product_Type
    mask = (
        (df_items_filtered['cat_3'].values == query_cat3)
        & (df_items_filtered['cat_4'].values != query_cat4)
        & (df_items_filtered['Product_Type'].values != query_product_type)
    )
    filtered_indices = np.where(mask)[0]

    if len(filtered_indices) == 0:
        print(f"No items found in cat_3='{query_cat3}' with a different cat_4 than '{query_cat4}' and different Product_Type than '{query_product_type}'.")
        return None

    # Cosine similarity against filtered items only
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm[filtered_indices].T

    # Top N
    top_n = min(n, len(scores))
    top_local = np.argsort(scores)[-top_n:][::-1]
    top_indices = filtered_indices[top_local]

    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_local].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'Product_Type': df_items_filtered.iloc[top_indices]['Product_Type'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })

    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin} | cat_3: {query_cat3} | cat_4: {query_cat4} | Product_Type: {query_product_type}")
    print(f"Same cat_3, different cat_4 & Product_Type items: {mask.sum():,}\n")

    return results

get_similar_items_same_cat3_diff_cat4_product_type('B00029TCRG', n=10)

In [ ]:
# Return the top N most similar items in different Cat_4 and Product_Type
def get_similar_items_diff_cat4_product_type(asin, n=10):
    """Return top N most similar items for a given ASIN, different Cat_4 and Product_Type."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]

    query_cat4 = df_items_filtered.iloc[idx]['cat_4']
    query_product_type = df_items_filtered.iloc[idx]['Product_Type']

    # Get indices of items with different Cat_4 and Product_Type
    mask = (
        (df_items_filtered['cat_4'].values != query_cat4)
        & (df_items_filtered['Product_Type'].values != query_product_type)
    )
    filtered_indices = np.where(mask)[0]

    if len(filtered_indices) == 0:
        print(f"No items found with a different Cat_4 than '{query_cat4}' and different Product_Type than '{query_product_type}'.")
        return None

    # Cosine similarity against filtered items only
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm[filtered_indices].T

    # Top N
    top_n = min(n, len(scores))
    top_local = np.argsort(scores)[-top_n:][::-1]
    top_indices = filtered_indices[top_local]

    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_local].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'Product_Type': df_items_filtered.iloc[top_indices]['Product_Type'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })

    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin} | Cat_4: {query_cat4} | Product_Type: {query_product_type}")
    print(f"Different Cat_4 & Product_Type items: {mask.sum():,}\n")

    return results

get_similar_items_diff_cat4_product_type('B00029TCRG', n=10)

In [ ]:
# Target item
pd.options.display.max_colwidth = None
print(f'target item:, {df_items_filtered[df_items_filtered['asin'] == 'B00029TCRG']['title']}')
print(f'similar item:, {df_items_filtered[df_items_filtered['asin'] == 'B00STIA3X4']['title']}')

In [ ]:
pd.options.display.max_colwidth = 50

In [ ]:
# Return the top N most similar items NOT in the same cat_3 category
def get_similar_items_diff_cat3(asin, n=10):
    """Return top N most similar items for a given ASIN, restricted to different cat_3."""
    idx = np.where(asins == asin)[0]
    if len(idx) == 0:
        print(f"ASIN '{asin}' not found.")
        return None
    idx = idx[0]

    query_cat3 = df_items_filtered.iloc[idx]['cat_3']

    # Get indices of items NOT in the same cat_3
    diff_cat3_mask = df_items_filtered['cat_3'].values != query_cat3
    diff_cat3_indices = np.where(diff_cat3_mask)[0]

    # Cosine similarity against different-category items only
    scores = embedding_matrix_norm[idx] @ embedding_matrix_norm[diff_cat3_indices].T

    # Top N
    top_n = min(n, len(scores))
    top_local = np.argsort(scores)[-top_n:][::-1]
    top_indices = diff_cat3_indices[top_local]

    results = pd.DataFrame({
        'asin': asins[top_indices],
        'similarity': scores[top_local].round(4),
        'title': df_items_filtered.iloc[top_indices]['title'].values,
        'title_cleaned': df_items_filtered.iloc[top_indices]['title_cleaned'].values,
        'cat_3': df_items_filtered.iloc[top_indices]['cat_3'].values,
        'cat_4': df_items_filtered.iloc[top_indices]['cat_4'].values,
        'brand': df_items_filtered.iloc[top_indices]['brand'].values,
        'extracted_features': df_items_filtered.iloc[top_indices]['extracted_features'].values
    })

    query_title = df_items_filtered.iloc[idx]['title_cleaned']
    print(f"Query: {query_title[:80]}")
    print(f"ASIN:  {asin} | cat_3: {query_cat3}")
    print(f"Different-category items: {diff_cat3_mask.sum():,}\n")

    return results

get_similar_items_diff_cat3('B00029TCRG', n=10)

## **Visualize Embeddings**

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl

# Bright, distinct colors for categories
BRIGHT_COLORS = [
    '#e6194B', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#800000', '#aaffc3',
    '#808000', '#ffd8b1', '#000075', '#a9a9a9', '#000000',
    '#e6beff', '#aa6e28', '#fffac8', '#00ff00', '#ff4500',
]

# Force white background
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

# Reduce SBERT embeddings to 2D with PCA
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(embedding_matrix_norm)

cat2_labels = df_items_filtered['cat_2'].values
unique_cat2 = sorted(set(cat2_labels))

fig, ax = plt.subplots(figsize=(14, 10))
fig.set_facecolor('white')
ax.set_facecolor('white')

for i, cat in enumerate(unique_cat2):
    mask = cat2_labels == cat
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
               s=1, alpha=0.15, label=cat, color=BRIGHT_COLORS[i % len(BRIGHT_COLORS)])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('SBERT Embeddings (PCA 2D) colored by cat_2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=8, fontsize=12,
          frameon=True, facecolor='white', edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
# Force white background
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

# Per-cat_2 PCA visualizations colored by cat_3
unique_cat2 = sorted(df_items_filtered['cat_2'].unique())

for cat2 in unique_cat2:
    cat2_mask = df_items_filtered['cat_2'].values == cat2
    cat2_indices = np.where(cat2_mask)[0]

    # PCA on this cat_2 subset
    pca_sub = PCA(n_components=2, random_state=42)
    embeddings_2d_sub = pca_sub.fit_transform(embedding_matrix_norm[cat2_indices])

    cat3_labels = df_items_filtered.iloc[cat2_indices]['cat_3'].values
    unique_cat3 = sorted(set(cat3_labels))

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.set_facecolor('white')
    ax.set_facecolor('white')

    for i, cat3 in enumerate(unique_cat3):
        mask = cat3_labels == cat3
        ax.scatter(embeddings_2d_sub[mask, 0], embeddings_2d_sub[mask, 1],
                   s=1, alpha=0.15, label=cat3, color=BRIGHT_COLORS[i % len(BRIGHT_COLORS)])

    ax.set_xlabel(f'PC1 ({pca_sub.explained_variance_ratio_[0]:.1%} variance)')
    ax.set_ylabel(f'PC2 ({pca_sub.explained_variance_ratio_[1]:.1%} variance)')
    ax.set_title(f'SBERT Embeddings (PCA 2D) — cat_2: {cat2} — colored by cat_3')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=8, fontsize=12,
              frameon=True, facecolor='white', edgecolor='black')
    plt.tight_layout()
    plt.show()

## **Standardize Features**

In [ ]:
pd.options.display.max_colwidth = None
df_items_filtered[(df_items_filtered['cat_3'] == 'Accent Furniture') &
                  (df_items_filtered['Dimensions'].isna() == False)][['category', 'title_cleaned', 'extracted_features',
                                                                      'cat_1', 'cat_2', 'cat_3', 'cat_4', 'cat_5', 'cat_6',
                                                                      'Dimensions']]

In [ ]:
df_items_filtered[(df_items_filtered['cat_3'] == 'Accent Furniture') &
                  (df_items_filtered['Dimensions'].isna() == False) &
                  (df_items_filtered['Dimensions'].str.count('x') == 2)][['category', 'title', 'title_cleaned', 'extracted_features',
                                                                          'cat_1', 'cat_2', 'cat_3', 'cat_4', 'cat_5', 'cat_6',
                                                                          'Dimensions']]

In [ ]:
df_items_filtered.head()

In [ ]:
####
####
#### Optional: The goal was for each of 1.3M items, return the top10 most similar ones
####
####
# Find top N most similar items for each item using dot product
N = 10  # hyperparameter: number of similar items to return

# Stack embeddings into a matrix
embedding_matrix = np.stack(df_features['title_embedding'].values)

# Normalize for cosine similarity via dot product
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
embedding_matrix_norm = embedding_matrix / norms

# Compute dot product in batches to avoid memory issues
batch_size = 1000
n_items = len(embedding_matrix_norm)
asins = df_features['asin'].values
titles = df_features['title_cleaned'].values

top_n_indices = []
top_n_scores = []

print(f"Computing top {N} similar items for {n_items:,} products...")
for start in range(0, n_items, batch_size):
    end = min(start + batch_size, n_items)
    # Dot product of batch against all items
    sim_batch = embedding_matrix_norm[start:end] @ embedding_matrix_norm.T
    
    # Zero out self-similarity
    for i in range(end - start):
        sim_batch[i, start + i] = -1
    
    # Get top N indices and scores
    top_indices = np.argsort(sim_batch, axis=1)[:, -N:][:, ::-1]
    top_scores = np.take_along_axis(sim_batch, top_indices, axis=1)
    
    top_n_indices.append(top_indices)
    top_n_scores.append(top_scores)
    
    if (start // batch_size) % 10 == 0:
        print(f"  Processed {end:,} / {n_items:,}")

top_n_indices = np.vstack(top_n_indices)
top_n_scores = np.vstack(top_n_scores)

# Store results
df_features['similar_items'] = [
    list(zip(asins[top_n_indices[i]], top_n_scores[i].round(4)))
    for i in range(n_items)
]

# Show samples
print(f"\nSample similar items:")
for idx in [0, 100, 500]:
    if idx >= n_items:
        break
    print(f"\n  Item: {titles[idx][:70]}")
    print(f"  ASIN: {asins[idx]}")
    for asin, score in df_features.iloc[idx]['similar_items'][:5]:
        match_title = df_features[df_features['asin'] == asin]['title_cleaned'].values[0][:60]
        print(f"    {score:.4f} | {asin} | {match_title}")

## **Analyze the purchases**

In [ ]:
# Number of times each item (asin) has been purchased/reviewed
purchase_counts = df_combined.groupby('asin').size().reset_index(name='purchase_count')

print(f"Total unique items: {len(purchase_counts):,}")
print(f"Mean purchases per item: {purchase_counts['purchase_count'].mean():.2f}")
print(f"Median purchases per item: {purchase_counts['purchase_count'].median():.0f}")
print(f"Items purchased only once: {(purchase_counts['purchase_count'] == 1).sum():,} ({(purchase_counts['purchase_count'] == 1).mean()*100:.1f}%)")

# All items
plt.figure(figsize=(12, 5))
plt.hist(purchase_counts['purchase_count'], bins=100, edgecolor='black', alpha=0.7)
plt.xlabel('Number of Purchases')
plt.ylabel('Number of Items')
plt.title('Distribution of Purchases per Item')
plt.tight_layout()
plt.show()

# Items with < 50 purchases
plt.figure(figsize=(12, 5))
subset = purchase_counts[purchase_counts['purchase_count'] < 50]
plt.hist(subset['purchase_count'], bins=49, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Number of Purchases')
plt.ylabel('Number of Items')
plt.title('Distribution of Purchases per Item (< 50 purchases)')
plt.tight_layout()
plt.show()

### **Update master_metadata.json**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Filter to valid cat_3 categories (more than 100 products to avoid noise)
cat3_counts_hd = df_correcting['cat_3'].value_counts()
valid_cat3_hd = cat3_counts_hd[cat3_counts_hd > 100].index.tolist()
df_correcting = df_correcting[df_correcting['cat_3'].isin(valid_cat3_hd)]

# Concatenate all title_cleaned texts per cat_3 into a single document per category
cat3_docs_hd = (
    df_correcting.groupby('cat_3')['title_cleaned']
    .apply(lambda x: ' '.join(x.dropna()))
    .reset_index()
)
cat3_docs_hd.columns = ['cat_3', 'text']

# Fit TF-IDF with unigrams, bigrams, and trigrams
tfidf_hd = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=50000,
    min_df=1,
    sublinear_tf=True
)
tfidf_matrix_hd = tfidf_hd.fit_transform(cat3_docs_hd['text'])
feature_names_hd = tfidf_hd.get_feature_names_out()

print(f"Valid cat_3 categories: {len(valid_cat3_hd)}")
print(f"TF-IDF matrix: {tfidf_matrix_hd.shape[0]} categories x {tfidf_matrix_hd.shape[1]} n-grams")

In [ ]:
# Combined: Frequency + TF-IDF for each cat_3
top_n = 100

for idx, row in cat3_docs_hd.iterrows():
    cat_name = row['cat_3']
    cat_count = cat3_counts_hd[cat_name]
    df_cat = df_correcting[df_correcting['cat_3'] == cat_name]
    titles = df_cat['title_cleaned'].dropna()
    all_words = ' '.join(titles).split()

    # --- Raw frequency ---
    unigram_counts = Counter(all_words)
    bigram_counts = Counter(zip(all_words, all_words[1:]))
    trigram_counts = Counter(zip(all_words, all_words[1:], all_words[2:]))

    # --- TF-IDF ---
    scores = tfidf_matrix_hd[idx].toarray().flatten()
    top_tfidf_indices = scores.argsort()[::-1][:top_n]

    print(f"\n{'='*70}")
    print(f"cat_3: {cat_name} ({cat_count} products)")
    print(f"{'='*70}")

    # Frequency tables
    print(f"\n--- Top {top_n} Unigrams (Frequency) ---")
    for word, count in unigram_counts.most_common(top_n):
        print(f"  {word:30s} {count:>8,}")

    print(f"\n--- Top {top_n} Bigrams (Frequency) ---")
    for bg, count in bigram_counts.most_common(top_n):
        print(f"  {bg[0]} {bg[1]:30s} {count:>8,}")

    print(f"\n--- Top {top_n} Trigrams (Frequency) ---")
    for tg, count in trigram_counts.most_common(top_n):
        print(f"  {tg[0]} {tg[1]} {tg[2]:30s} {count:>8,}")

    # TF-IDF table
    print(f"\n--- Top {top_n} N-grams (TF-IDF) ---")
    print(f"  {'Rank':<6} {'TF-IDF':>8}  {'N-gram'}")
    print(f"  {'-'*6} {'-'*8}  {'-'*40}")
    for rank, i in enumerate(top_tfidf_indices, 1):
        if scores[i] > 0:
            print(f"  {rank:<6} {scores[i]:>8.4f}  {feature_names_hd[i]}")

### **Visualize Word Count**

In [ ]:
# Word count for title, description, and feature
df_items['title_word_count'] = df_items['title'].fillna('').str.split().str.len()
df_items['description_word_count'] = df_items['description'].fillna('').str.split().str.len()
df_items['feature_word_count'] = df_items['feature'].fillna('').str.split().str.len()

df_items[['asin', 'title_word_count', 'description_word_count', 'feature_word_count']].describe()

In [ ]:
df_items[['asin', 'title', 'title_word_count', 'description_word_count', 'feature_word_count']].sort_values('title_word_count',ascending=False).head(20)


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_items['description_word_count'], bins=50, edgecolor='black')
plt.xlabel('Word Count')
plt.ylabel('Number of Products')
plt.title('Distribution of Description Word Count')
plt.xticks(range(0, int(df_items['description_word_count'].max()) + 500, 500))
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_items['feature_word_count'], bins=50, edgecolor='black')
plt.xlabel('Word Count')
plt.ylabel('Number of Products')
plt.title('Distribution of Feature Word Count')
plt.xticks(range(0, int(df_items['feature_word_count'].max()) + 200, 200))
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_items['title_word_count'], bins=50, edgecolor='black')
plt.xlabel('Word Count')
plt.ylabel('Number of Products')
plt.title('Distribution of Title Word Count')
plt.xticks(range(0, int(df_items['title_word_count'].max()) + 100, 100))
plt.show()

**Categories**

In [ ]:
df_items[df_items['cat_2'] == 'Furniture']['cat_3'].value_counts()
# df_ items['cat_2'].value_counts()